In [ ]:
GITHUB_REPO = "https://github.com/minhE10/time-series-prediction.git"

CLONE_DIR = "/kaggle/working/ts-project"                       

# arima | itransformer | timemixer | tsmamba | xgboost
MODEL_NAME = "timemixer"

# sunspots | appliances_energy | beijing_air_quality | hanoi_air_quality | bitcoin
DATA_NAME = "bitcoin"
SEQ_LEN = 48.          # lookback window (fixed)
PRED_LEN = 24.         # forecast horizon: 24 | 48 | 72
NUM_EPOCHS = 200         
SCALER = "standard"    # standard | minmax

# change to the right path
DATASET_DIR = "/kaggle/input/"
DATASET_PATHS = {
    "sunspots": f"{DATASET_DIR}/sunspots_dataset.csv",
    "appliances_energy": f"{DATASET_DIR}/appliances_energy_dataset.csv",
    "beijing_air_quality": f"{DATASET_DIR}/beijing_air_quality_dataset.csv",
    "hanoi_air_quality": f"{DATASET_DIR}/hanoi_air_quality_dataset.csv",
    "bitcoin": f"{DATASET_DIR}/bitcoin_dataset.csv",
}

In [ ]:
import subprocess, sys
subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, CLONE_DIR], check=True)
sys.path.insert(0, CLONE_DIR)

from setup import set_seed, clear_memory
SEED_WORKER, G, DEVICE = set_seed(42)

In [ ]:
from data_loader import TimeSeriesDataset

dataset = TimeSeriesDataset(
    name=DATA_NAME, path=DATASET_PATHS[DATA_NAME],
    seq_len=SEQ_LEN, pred_len=PRED_LEN,
    batch_size=32, worker_init_fn=SEED_WORKER, generator=G,
)
dataset.apply_scaling(SCALER)
train_loader, val_loader, test_loader = dataset.get_loaders()

print(f"Dataset : {DATA_NAME}  |  Features: {dataset.n_features}  Targets: {dataset.n_targets}")
print(f"Windows : train={len(dataset.train_dataset)}  val={len(dataset.val_dataset)}  test={len(dataset.test_dataset)}")

In [ ]:
from src.factory import build_model, build_trainer

model   = build_model(MODEL_NAME, dataset, SEQ_LEN, PRED_LEN)
trainer = build_trainer(MODEL_NAME, model, dataset, train_loader, val_loader, test_loader, DEVICE)

if model is not None:
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {MODEL_NAME}  |  Parameters: {n_params:,}")

In [ ]:
history = trainer.fit(NUM_EPOCHS)

In [ ]:
from plot import plot_loss, plot_predictions

trainer.evaluate(scaled=False, title=f"{MODEL_NAME.upper()} — {DATA_NAME}")

plot_loss(history, title=f"{MODEL_NAME.upper()} on {DATA_NAME}")

y_true, y_pred = trainer.predict()
plot_predictions(y_true, y_pred, dataset.target_cols,
                 title=f"{MODEL_NAME.upper()} on {DATA_NAME} (Test)")